# EX_06 — Introducción a RAG (ejercicios)

**Notebook de referencia:** `notebook/06_Introduccion_RAG.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Plantilla de contexto

Escribe una función `build_prompt(context_chunks, question) -> str` que inserte los pasajes en un delimitador claro (`### Context` / `### Question`).


In [6]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    # 1. Unimos todos los chunks recuperados separándolos por un doble salto de línea
    full_context = "\n\n".join(context_chunks)
    
    # 2. Construimos la plantilla utilizando cadenas multilínea (f-strings)
    prompt = f"""### Context
{full_context}

### Question
{question}
"""
    return prompt
print(build_prompt(["Chunk 1: Información relevante.", "Chunk 2: Más detalles."], "¿Cuál es la información más importante?"))

### Context
Chunk 1: Información relevante.

Chunk 2: Más detalles.

### Question
¿Cuál es la información más importante?



## Actividad 2 — RAG sin LLM (retrieval only)

Con tus chunks del notebook teórico (o texto inventado), recupera top-k y **imprime** el contexto ensamblado sin llamar al generador.


In [3]:
# 1. Definimos una pregunta fija y unos chunks simulados (puedes usar los de tus actividades previas)
fixed_question = "¿Cómo procesan la información los Transformers?"
mis_chunks_en_base_datos = [
    "Los Transformers procesan secuencias completas en paralelo usando mecanismos de auto-atención.",
    "El mecanismo de atención calcula la relevancia mutua entre cada par de palabras de un texto.",
    "Las redes neuronales recurrentes (RNN) procesaban los textos palabra por palabra de forma secuencial."
]

# 2. Simulamos la recuperación del Top-K (por ejemplo, los 2 primeros fragmentos más relevantes)
top_k_chunks = mis_chunks_en_base_datos[:2]

# 3. Ensamblamos el contexto utilizando la función de la Actividad 1
ensamblado = build_prompt(context_chunks=top_k_chunks, question=fixed_question)

# 4. Imprimimos el resultado en pantalla sin llamar a ningún LLM
print(ensamblado)


### Context
Los Transformers procesan secuencias completas en paralelo usando mecanismos de auto-atención.

El mecanismo de atención calcula la relevancia mutua entre cada par de palabras de un texto.

### Question
¿Cómo procesan la información los Transformers?



## Actividad 3 — Fallo de cobertura

Inventa un caso donde la respuesta **no** está en los chunks recuperados y describe en español (markdown) cómo lo detectarías en producción (p. ej. umbral de score, abstención).


**Caso inventado de fallo de cobertura:**
Pregunta del usuario: "¿Cuál es la política de devoluciones para productos comprados durante el Black Friday de 2025?"

Contenido de los chunks recuperados: El recuperador vectoriza la frase y extrae fragmentos genéricos sobre "Políticas de devoluciones generales (30 días)" y "Cómo solicitar un reembolso en la web", pero ningún fragmento menciona las condiciones específicas ni las excepciones aplicadas a las ofertas del Black Friday de 2025 porque ese documento no existe en la base de datos o no se indexó correctamente.

Cómo detectarlo en producción:
Para mitigar que el modelo alucine inventándose la política de devoluciones, se pueden implementar dos estrategias automatizadas en producción:

Umbral de puntuación mínimo (Score Thresholding):
Al buscar en la base de datos vectorial (como FAISS, Chroma o Pinecone), cada fragmento se recupera con una distancia o score de similitud. Si el mejor fragmento del Top-K tiene un score por debajo de un umbral de seguridad crítico establecido (por ejemplo, una similitud coseno inferior a 0.45), el sistema clasifica automáticamente la recuperación como inválida o "no confiable".

Instrucción de Abstención en el System Prompt:
Se configura de manera estricta el prompt del sistema para forzar el comportamiento del LLM mediante reglas de control. Se le añade la directiva: "Si la información necesaria para responder a la pregunta no se encuentra explícitamente dentro del bloque ### Context, debes abstenerte de responder y contestar textualmente: 'Lo siento, no dispongo de información suficiente en mis documentos para responder a tu pregunta'." Mediante la combinación de ambas métricas (filtro de score en código + regla de abstención en el LLM), evitamos de forma robusta las alucinaciones en entornos reales._Tu explicación:_

...
